# Phase 1 — Data Pipeline

This notebook demonstrates the complete data engineering workflow using the production code in `src/fx_forecast`.

## 1. Import production components

In [1]:
import pandas as pd

from fx_forecast.config.settings import settings
from fx_forecast.config.paths import PROCESSED_DATA_DIR
from fx_forecast.data.pipeline import run_pipeline
from fx_forecast.data.preprocess import preprocess_dataframe
from fx_forecast.data.providers.yahoo import YahooFXProvider
from fx_forecast.data.validate import validate_dataframe

## 2. Inspect application settings

In [2]:
print("Currency pairs:", settings.currency_pairs)
print("Start date:", settings.start_date)
print("Interval:", settings.interval)
print("Random seed:", settings.random_seed)

Currency pairs: ['USD/NGN', 'EUR/NGN']
Start date: 2015-01-01
Interval: 1d
Random seed: 42


## 3. Select a provider

In [ ]:
provider = YahooFXProvider(
    interval=settings.interval,
)

pair = settings.currency_pairs[0]

print("Provider:", provider.__class__.__name__)
print("Pair:", pair)
print("Schema:", provider.schema)

Provider: YahooFXProvider
Pair: USD/NGN
Schema: DataSchema(required_columns=('Open', 'High', 'Low', 'Close', 'Volume'), numeric_columns=('Open', 'High', 'Low', 'Close', 'Volume'))


## 4. Fetch FX data

In [4]:
raw_df = provider.fetch(
    pair=pair,
    start=settings.start_date,
)

raw_df.head()

,Close,High,Low,Open,Volume
Date,,,,,
2015-01-01,182.300003,181.380005,181.380005,181.380005,0
2015-01-02,181.380005,182.899994,181.009995,181.869995,0
2015-01-05,182.699997,183.399994,181.500000,182.350006,0
2015-01-06,183.199997,181.949997,180.979996,180.979996,0
2015-01-07,183.600006,182.279999,181.350006,182.279999,0


## 5. Inspect raw data

In [5]:
raw_df.info()

raw_df.describe().T

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 3022 entries, 2015-01-01 to 2026-08-12
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Close   3022 non-null   float64
 1   High    3022 non-null   float64
 2   Low     3022 non-null   float64
 3   Open    3022 non-null   float64
 4   Volume  3022 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 141.7 KB


,count,mean,std,min,25%,50%,75%,max
Close,3022.0,616.802081,478.236194,1.000000,356.0,380.500000,775.644989,1696.199951
High,3022.0,620.854568,479.827479,180.419998,358.0,385.890015,780.000000,1702.229980
Low,3022.0,614.370476,475.633524,1.000000,355.0,380.000000,767.202515,1684.800049
Open,3022.0,617.522880,477.761170,175.669998,357.0,380.700012,773.000000,1692.640015
Volume,3022.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000


## 6. Validate provider-specific data

In [6]:
validated_df = validate_dataframe(
    raw_df,
    schema=provider.schema,
)

validated_df.head()

2026-08-13 01:40:39 | SUCCESS  | fx_forecast.data.validate:validate_dataframe:54 | Dataset validation passed.


,Close,High,Low,Open,Volume
Date,,,,,
2015-01-01,182.300003,181.380005,181.380005,181.380005,0
2015-01-02,181.380005,182.899994,181.009995,181.869995,0
2015-01-05,182.699997,183.399994,181.500000,182.350006,0
2015-01-06,183.199997,181.949997,180.979996,180.979996,0
2015-01-07,183.600006,182.279999,181.350006,182.279999,0


## 7. Preprocess the data

In [7]:
processed_df = preprocess_dataframe(validated_df)

processed_df.head()

2026-08-13 01:40:40 | SUCCESS  | fx_forecast.data.preprocess:preprocess_dataframe:33 | Preprocessing complete (3022 → 3022 rows).


,Close,High,Low,Open,Volume
Date,,,,,
2015-01-01,182.300003,181.380005,181.380005,181.380005,0
2015-01-02,181.380005,182.899994,181.009995,181.869995,0
2015-01-05,182.699997,183.399994,181.500000,182.350006,0
2015-01-06,183.199997,181.949997,180.979996,180.979996,0
2015-01-07,183.600006,182.279999,181.350006,182.279999,0


## 8. Run the complete production pipeline

In [8]:
pipeline_df = run_pipeline(
    provider=provider,
    pair=pair,
    start=settings.start_date,
)

pipeline_df.head()

2026-08-13 01:40:40 | INFO     | fx_forecast.data.pipeline:run_pipeline:23 | Starting pipeline for USD/NGN
2026-08-13 01:40:41 | SUCCESS  | fx_forecast.data.validate:validate_dataframe:54 | Dataset validation passed.
2026-08-13 01:40:41 | SUCCESS  | fx_forecast.data.preprocess:preprocess_dataframe:33 | Preprocessing complete (3022 → 3022 rows).
2026-08-13 01:40:41 | SUCCESS  | fx_forecast.data.io:save_dataframe:56 | Saved dataset -> C:\Engineering\02_Projects\fx-forecast-system\data\processed\USD_NGN.csv
2026-08-13 01:40:41 | SUCCESS  | fx_forecast.data.pipeline:run_pipeline:45 | Pipeline completed for USD/NGN


,Close,High,Low,Open,Volume
Date,,,,,
2015-01-01,182.300003,181.380005,181.380005,181.380005,0
2015-01-02,181.380005,182.899994,181.009995,181.869995,0
2015-01-05,182.699997,183.399994,181.500000,182.350006,0
2015-01-06,183.199997,181.949997,180.979996,180.979996,0
2015-01-07,183.600006,182.279999,181.350006,182.279999,0


## 9. Verify processed output

In [9]:
output_path = (
    PROCESSED_DATA_DIR
    / f"{pair.replace('/', '_')}.csv"
)

print(output_path)
print(output_path.exists())

pipeline_df.info()

C:\Engineering\02_Projects\fx-forecast-system\data\processed\USD_NGN.csv
True
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 3022 entries, 2015-01-01 to 2026-08-12
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Close   3022 non-null   float64
 1   High    3022 non-null   float64
 2   Low     3022 non-null   float64
 3   Open    3022 non-null   float64
 4   Volume  3022 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 141.7 KB



# Phase 1 Summary

- Provider-based FX data ingestion
- Provider-specific schema validation
- Data preprocessing
- Processed dataset persistence
- End-to-end production pipeline execution

The Phase 1 data engineering layer is now ready for Feature Engineering.